# Phase 1: Auditable Incident Deduplication

This notebook is the analyst-facing view of the Phase 1 v3 pipeline. It preserves every source report and assigns each configured same-crossing comparison pair exactly one deterministic decision: `auto_merge` or `keep_distinct`.

Duration categories are documented proxy intervals, not measured endpoints. `Date/Time` is interpreted as UTC under the approved source contract. Crossing-local timestamps are derived from the current Form 71 coordinate and historical IANA daylight-saving rules; neither field establishes physical-event truth.

In [1]:
import hashlib
import importlib
import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# -------------------------------------------------------------------------
# Workflow Configuration Switches
# -------------------------------------------------------------------------
REUSE_STEP_5_CHECKPOINT = False
RUN_REPEATABILITY_CHECK = False

# Repeatability mode takes precedence and disables checkpoint reuse
if RUN_REPEATABILITY_CHECK:
    REUSE_STEP_5_CHECKPOINT = False

repo_root = Path(r"C:/Projects/Blocked-Crossing-Prediction")
analysis_dir = (repo_root / "analysis").resolve()
analysis_path = str(analysis_dir)
sys.path = [entry for entry in sys.path if str(Path(entry or '.').resolve()) != analysis_path]
sys.path.insert(0, analysis_path)

sys.modules.pop("incident_deduplication", None)
import incident_deduplication
incident_deduplication = importlib.reload(incident_deduplication)
expected_module_path = (analysis_dir / "incident_deduplication.py").resolve()
loaded_module_path = Path(incident_deduplication.__file__).resolve()
if loaded_module_path != expected_module_path:
    raise RuntimeError(f"Loaded {loaded_module_path}, expected {expected_module_path}. Restart the kernel and rerun this cell.")
if not hasattr(incident_deduplication, "generate_pair_decisions"):
    raise RuntimeError("The loaded incident_deduplication module is not the Phase 1 v3 implementation. Reload the VS Code window, reopen the notebook, restart the kernel, and rerun this cell.")
run_phase_1 = incident_deduplication.run_phase_1

authoritative_path = repo_root / "data" / "blocked_crossings_2020through2025.xlsx"
reconciliation_path = repo_root / "data" / "blocked_crossings_2025.xlsx"
inventory_path = repo_root / "data" / "Crossing_Inventory_Data_(Form_71)_-_Current_20260707.csv"
config_path = analysis_dir / "incident_deduplication_config.json"
output_dir = repo_root / "analysis_outputs" / "deduplication" / "v3"
config_preview = json.loads(config_path.read_text(encoding="utf-8"))
if config_preview.get("ruleset_version") != "phase1-v3" or "pair_decision_bands_minutes" not in config_preview:
    raise RuntimeError("Phase 1 configuration is not compatible with the v3 pair-decision pipeline.")
print(f"Loaded Phase 1 v3 module: {loaded_module_path}")

# Record initial SHA-256 hashes strictly for raw input integrity verification
def compute_file_hash(filepath: Path) -> str:
    hasher = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(8192):
            hasher.update(chunk)
    return hasher.hexdigest()

initial_input_hashes = {
    "authoritative": compute_file_hash(authoritative_path),
    "reconciliation": compute_file_hash(reconciliation_path),
    "inventory": compute_file_hash(inventory_path),
}

# -------------------------------------------------------------------------
# Pipeline Execution Logic
# -------------------------------------------------------------------------
if RUN_REPEATABILITY_CHECK:
    print("Running Repeatability Diagnostic Mode (Two Fresh Runs)...")
    
    # Run 1
    dir_run1 = output_dir / "repeatability_run1"
    result_run1 = run_phase_1(
        authoritative_path,
        reconciliation_path,
        inventory_path,
        config_path,
        dir_run1,
        reuse_step_5_checkpoint=False,
    )
    
    # Run 2
    dir_run2 = output_dir / "repeatability_run2"
    result_run2 = run_phase_1(
        authoritative_path,
        reconciliation_path,
        inventory_path,
        config_path,
        dir_run2,
        reuse_step_5_checkpoint=False,
    )
    
    # Primary result set for notebook display
    result = result_run1
    
    # Compare summary metrics between independent runs
    two_real_data_runs_match = result_run1.summary == result_run2.summary
    result.summary["repeatability_check"] = "passed" if two_real_data_runs_match else "failed"

else:
    # Single run mode (Fresh execution or Resumed from Step 5 Checkpoint)
    result = run_phase_1(
        authoritative_path,
        reconciliation_path,
        inventory_path,
        config_path,
        output_dir,
        reuse_step_5_checkpoint=REUSE_STEP_5_CHECKPOINT,
    )
    result.summary["repeatability_check"] = "not_run"

# Prove raw inputs were not mutated during pipeline execution
post_input_hashes = {
    "authoritative": compute_file_hash(authoritative_path),
    "reconciliation": compute_file_hash(reconciliation_path),
    "inventory": compute_file_hash(inventory_path),
}
assert initial_input_hashes == post_input_hashes, "Raw input files were modified during execution!"

# Output Summary and Immediate Validations
print(json.dumps(result.summary, indent=2))

assert result.validations["source_rows_map_once"]
assert result.validations["only_configured_auto_merge_tiers"]

if RUN_REPEATABILITY_CHECK:
    assert result.summary.get("repeatability_check") == "passed", "Repeatability check failed between the two runs."

[1/9] Initializing output directory and checking input files...
[2/9] Reading raw source data files (Excel & CSV)...
      -> Loaded raw datasets in 21.9s
[3/9] Normalizing source data structures...
[4/9] Resolving crossing time zones (timezonefinder)...
      -> Resolved time zones in 322.5s
[5/9] Enriching local time and consolidating incident reports...
      -> Enriched local time in 10.0s
      -> Consolidated 135,135 source reports into 132,240 incidents in 11.2s
      -> Wrote step-5 checkpoint in 2.0s
[6/9] Assigning deterministic pair decisions...
      -> Assigned 57,282 pair decisions and produced 125,673 incidents in 91.3s
[7/9] Sampling decision-audit cases & running 2025 reconciliation...
[8/9] Validating dataset integrity checks...
[9/9] Writing output artifacts, gate report, and run manifest...
      -> Artifacts & manifest written in 8.6s
=== Phase 1 finished successfully in 543.7s ===
{
  "ruleset_version": "phase1-v3",
  "source_reports": 135135,
  "candidate_reporte

In [2]:
result = run_phase_1(
    authoritative_path, reconciliation_path, inventory_path, config_path, output_dir
)
print(json.dumps(result.summary, indent=2))
assert result.validations["source_rows_map_once"]
assert result.validations["only_configured_auto_merge_tiers"]

[1/9] Initializing output directory and checking input files...
[2/9] Reading raw source data files (Excel & CSV)...
      -> Loaded raw datasets in 28.6s
[3/9] Normalizing source data structures...
[4/9] Resolving crossing time zones (timezonefinder)...
      -> Resolved time zones in 303.6s
[5/9] Enriching local time and consolidating incident reports...
      -> Enriched local time in 4.8s
      -> Consolidated 135,135 source reports into 132,240 incidents in 6.4s
      -> Wrote step-5 checkpoint in 1.9s
[6/9] Assigning deterministic pair decisions...
      -> Assigned 57,282 pair decisions and produced 125,673 incidents in 56.6s
[7/9] Sampling decision-audit cases & running 2025 reconciliation...
[8/9] Validating dataset integrity checks...
[9/9] Writing output artifacts, gate report, and run manifest...
      -> Artifacts & manifest written in 3.7s
=== Phase 1 finished successfully in 510.7s ===
{
  "ruleset_version": "phase1-v3",
  "source_reports": 135135,
  "candidate_reported_

## Inventory, normalization, and provenance

Unknown duration values remain unmapped; invalid crossing IDs and timestamps remain in the source table and receive documented exceptions rather than canonical incidents.

In [2]:
inventory_profile = json.loads((output_dir / 'inventory_profile.json').read_text(encoding='utf-8'))
pd.DataFrame([inventory_profile['duration_normalization'], inventory_profile['crossing_id_status']], index=['duration status', 'crossing ID status']).T.fillna(0)

,duration status,crossing ID status
canonical,135133.0,0.0
known_alias,2.0,0.0
valid,0.0,135131.0
invalid_format,0.0,4.0


## UTC and crossing-local time

UTC remains the canonical comparison timestamp. Rows without a coordinate-derived IANA zone remain UTC-only and are flagged; no state-level fallback is used.

In [3]:
timezone_coverage = pd.read_csv(output_dir / 'timezone_assignment_diagnostics.csv')
local_time_diagnostics = pd.read_csv(output_dir / 'local_time_diagnostics.csv')
display(timezone_coverage)
local_time_diagnostics.sort_values(['iana_time_zone', 'reported_local_hour']).head(30)

,timezone_assignment_status,source_report_count
0,assigned,135131
1,invalid_crossing_id,4


,iana_time_zone,reported_local_hour,source_report_count
0,America/Anchorage,7,1
1,America/Anchorage,8,1
2,America/Anchorage,9,3
3,America/Anchorage,11,1
4,America/Anchorage,12,8
5,America/Anchorage,13,2
6,America/Anchorage,14,1
7,America/Anchorage,17,1
8,America/Anchorage,18,1
9,America/Boise,0,5


In [4]:
source_reports = pd.read_parquet(output_dir / 'source_reports_with_ids.parquet')
source_reports.loc[source_reports['timezone_assignment_status'].eq('assigned'), [
    'source_row_id', 'norm_crossing_id', 'reported_at_utc', 'reported_at_local',
    'iana_time_zone', 'utc_offset_minutes', 'State', 'City'
]].head(20)

,source_row_id,norm_crossing_id,reported_at_utc,reported_at_local,iana_time_zone,utc_offset_minutes,State,City
0,SRC-dfaab4c29680aee2661f,868252B,2022-09-09 16:55:00+00:00,2022-09-09T08:55:00-0800,America/Anchorage,-480,AK,ANCHORAGE
1,SRC-c8051fb600fa4f1bab24,868287C,2024-07-11 01:15:00+00:00,2024-07-10T17:15:00-0800,America/Anchorage,-480,AK,WHITTIER
2,SRC-c7e140d67e23efadfb01,868292Y,2025-09-20 17:40:00+00:00,2025-09-20T09:40:00-0800,America/Anchorage,-480,AK,WHITTIER
3,SRC-a8c701b24f8ef0e9d1d1,868292Y,2025-09-10 21:15:00+00:00,2025-09-10T13:15:00-0800,America/Anchorage,-480,AK,WHITTIER
4,SRC-1184e7c82762eda268ef,868292Y,2025-07-25 22:39:00+00:00,2025-07-25T14:39:00-0800,America/Anchorage,-480,AK,WHITTIER
5,SRC-9d96b0c83a2fe701f239,868292Y,2024-09-28 15:00:00+00:00,2024-09-28T07:00:00-0800,America/Anchorage,-480,AK,WHITTIER
6,SRC-b945471be013e5be93e7,868292Y,2024-08-21 20:52:00+00:00,2024-08-21T12:52:00-0800,America/Anchorage,-480,AK,WHITTIER
7,SRC-e71ddef0e37020407dac,868292Y,2024-08-01 17:36:00+00:00,2024-08-01T09:36:00-0800,America/Anchorage,-480,AK,WHITTIER
8,SRC-e16eba9ce131d3b9d567,868292Y,2024-05-30 17:15:00+00:00,2024-05-30T09:15:00-0800,America/Anchorage,-480,AK,WHITTIER
9,SRC-488e3836724b27defb25,868292Y,2023-07-06 21:10:00+00:00,2023-07-06T13:10:00-0800,America/Anchorage,-480,AK,WHITTIER


## Deterministic pair decisions and uncertainty

Every configured comparison pair is decided as `auto_merge` or `keep_distinct`. `decision_basis` records why, while `uncertainty_flag` and `uncertainty_basis` describe evidence strength without creating an unresolved workflow state. A `possible_temporal_overlap` flag is proxy evidence, not proof that two reports describe the same physical event.

In [5]:
deduplication_summary = json.loads((output_dir / 'deduplication_summary.json').read_text(encoding='utf-8'))
display(pd.Series(deduplication_summary))
pair_decision_summary = json.loads((output_dir / 'pair_decision_summary.json').read_text(encoding='utf-8'))
display(pd.Series(pair_decision_summary))
pair_decisions = pd.read_parquet(output_dir / 'pair_decisions.parquet')
decision_audit_sample = pd.read_csv(output_dir / 'pair_decision_audit_sample.csv')
decision_audit_sample.head(20)

ruleset_version                                                         phase1-v3
source_reports                                                             135135
candidate_reported_incidents                                               125673
exceptions                                                                      4
auto_merged_reports                                                          9458
consolidation_tiers             {'distinct_candidate': 120051, 'overlap_proxy_...
pair_decisions                                                              57282
pair_decision_counts                {'keep_distinct': 44930, 'auto_merge': 12352}
decision_basis_counts           {'overlap_proxy_but_non_temporal_mismatch': 34...
uncertain_pair_decisions                                                    48959
decision_audit_sample_rows                                                    341
timezone_assignment                {'assigned': 135131, 'invalid_crossing_id': 4}
dtype: object

pair_count                                                           57282
decision_counts              {'keep_distinct': 44930, 'auto_merge': 12352}
decision_basis_counts    {'overlap_proxy_but_non_temporal_mismatch': 34...
uncertainty_counts       {'possible_temporal_overlap': 48959, 'none': 8...
audit_sample_rows                                                      341
manual_labels_used                                                   False
dtype: object

,pair_decision_id,left_report_group_id,right_report_group_id,norm_crossing_id,separation_minutes,proximity_band_minutes,crossing_volume_tier,left_reported_at_utc,right_reported_at_utc,left_reported_at_local,...,left_railroad,right_railroad,left_reason,right_reason,left_immediate_impacts,right_immediate_impacts,left_additional_comments,right_additional_comments,left_final_incident_id,right_final_incident_id
0,PAIR-3abbe3d58316d124fb5e,INC-88a33df623caa4a93517,INC-cb0c8329b486d76b9750,260689A,6.000000,15,high,2020-07-05 05:24:00+00:00,2020-07-05 05:30:00+00:00,2020-07-05T00:24:00-0500,...,WC,WC,A stationary train,A stationary train,First responders were observed being unable to...,First responders were observed being unable to...,Fuck you and fuck off,Fuck you and fuck off,INC-98dfbf9c42754867d60a,INC-80a4cb2353253dd11ca8
1,PAIR-d5e3fe662c693b566600,INC-01d15b384f54cb867ec5,INC-576bd7b0df060b22cc55,260689A,14.000000,15,high,2020-07-05 05:12:00+00:00,2020-07-05 05:26:00+00:00,2020-07-05T00:12:00-0500,...,WC,WC,A stationary train,A stationary train,First responders were observed being unable to...,First responders were observed being unable to...,Fuck you and fuck off,Fuck you and fuck off,INC-98dfbf9c42754867d60a,INC-80a4cb2353253dd11ca8
2,PAIR-595a739464ae1f685606,INC-64ea451c909cc07f6d2b,INC-01d15b384f54cb867ec5,260689A,7.000000,15,high,2020-07-05 05:05:00+00:00,2020-07-05 05:12:00+00:00,2020-07-05T00:05:00-0500,...,WC,WC,A stationary train,A stationary train,First responders were observed being unable to...,First responders were observed being unable to...,Fuck you and fuck off,Fuck you and fuck off,INC-dfcae499c94c5b09bc9a,INC-98dfbf9c42754867d60a
3,PAIR-48c111f8a220c0694183,INC-e9ae683955b4def9aaae,INC-8c193373e93124655880,156099V,53.000000,60,high,2020-03-12 05:39:00+00:00,2020-03-12 06:32:00+00:00,2020-03-12T00:39:00-0500,...,NS,NS,A stationary train,A stationary train,First responders were observed being unable to...,First responders were observed being unable to...,NaN,NaN,INC-2b9a20bd77845faed8c9,INC-9e8a052e589adbf572dc
4,PAIR-c65f7cf2597ed1fd1817,INC-676d1a56536b937a80a4,INC-01d15b384f54cb867ec5,260689A,10.000000,15,high,2020-07-05 05:02:00+00:00,2020-07-05 05:12:00+00:00,2020-07-05T00:02:00-0500,...,WC,WC,A stationary train,A stationary train,First responders were observed being unable to...,First responders were observed being unable to...,Fuck you and fuck off,Fuck you and fuck off,INC-dfcae499c94c5b09bc9a,INC-98dfbf9c42754867d60a
5,PAIR-998e463fc494085ccaea,INC-b6434b85243b7a736aa1,INC-c857ff7650fcd8d66275,060399P,18.000000,30,high,2025-11-06 16:50:00+00:00,2025-11-06 17:08:00+00:00,2025-11-06T09:50:00-0700,...,BNSF,BNSF,A stationary train,A stationary train,NaN,NaN,NaN,NaN,INC-8d550a6cbcdcef933027,INC-490c687bd4679ef09e9c
6,PAIR-3d0479d5ae171c8bb3c1,INC-ed4de99dd780773d493a,INC-01d15b384f54cb867ec5,260689A,8.000000,15,high,2020-07-05 05:04:00+00:00,2020-07-05 05:12:00+00:00,2020-07-05T00:04:00-0500,...,WC,WC,A stationary train,A stationary train,First responders were observed being unable to...,First responders were observed being unable to...,Fuck you and fuck off,Fuck you and fuck off,INC-dfcae499c94c5b09bc9a,INC-98dfbf9c42754867d60a
7,PAIR-96a98b7c0f360e3cdaf2,INC-22c7e6a61c74c2e5fb79,INC-24387315f3b13d3e71e9,481481W,13.000000,15,high,2022-04-27 10:17:00+00:00,2022-04-27 10:30:00+00:00,2022-04-27T06:17:00-0400,...,NS,NS,A moving train,A moving train,NaN,NaN,NaN,NaN,INC-e210c6c0c22d0e452db7,INC-8bd63af470880d94aa09
8,PAIR-0f2d22ee2b0610c05e62,INC-2ebc4e5ed16cb82baf9c,INC-c2d5faf0bb5710b4e570,156099V,27.000000,30,high,2020-03-12 06:06:00+00:00,2020-03-12 06:33:00+00:00,2020-03-12T01:06:00-0500,...,NS,NS,A stationary train,A stationary train,First responders were observed being unable to...,First responders were observed being unable to...,NaN,NaN,INC-2b9a20bd77845faed8c9,INC-9e8a052e589adbf572dc
9,PAIR-e6a846b70084e75aa504,INC-291bb494b246854bcded,INC-547a662c601992e09756,263984P,112.600000,120,high,2022-04-07 1

## Reconciliation, diagnostics, and gate

The 2025 comparison is a normalized full-row multiset comparison, including duplicate multiplicity. A no-report interval must not be called unblocked.

In [6]:
reconciliation_summary = json.loads((output_dir / 'reconciliation_summary.json').read_text(encoding='utf-8'))
gate_report = json.loads((output_dir / 'phase_1_gate_report.json').read_text(encoding='utf-8'))
display(pd.Series(reconciliation_summary))
display(pd.read_csv(output_dir / 'timestamp_granularity_by_year.csv'))
display(pd.read_csv(output_dir / 'diagnostics_by_year.csv'))
gate_report

comparison                                        normalized full-row multiset over all configur...
authoritative_2025_rows                                                                       26186
authoritative_2025_unique_signatures                                                          25700
reconciliation_rows                                                                           26131
reconciliation_unique_signatures                                                              25646
rows_present_only_in_authoritative                                                               55
rows_present_only_in_reconciliation                                                               0
unique_signatures_with_multiplicity_difference                                                    0
dtype: object

,year,total_rows,missing_or_invalid_values,nonzero_seconds_count,nonzero_seconds_percentage,five_minute_mark_count,five_minute_mark_percentage,fifteen_minute_mark_count,fifteen_minute_mark_percentage,thirty_minute_mark_count,thirty_minute_mark_percentage,sixty_minute_mark_count,sixty_minute_mark_percentage,minimum_timestamp_utc,maximum_timestamp_utc
0,2020,10402,0,2658,25.5528,7207,69.2848,4344,41.7612,2851,27.4082,1761,16.9294,2020-01-02 07:30:00+00:00,2020-12-31 18:48:42+00:00
1,2021,21663,0,3961,18.2846,13440,62.0413,9277,42.8242,6779,31.2930,4660,21.5113,2021-01-01 04:08:00+00:00,2021-12-31 23:50:00+00:00
2,2022,30799,0,2236,7.2600,16328,53.0147,10373,33.6797,7291,23.6728,4604,14.9485,2022-01-01 00:00:00+00:00,2022-12-31 22:50:00+00:00
3,2023,19329,0,37,0.1914,10321,53.3965,6303,32.6090,4345,22.4792,2704,13.9893,2023-01-01 01:10:00+00:00,2023-12-31 22:45:00+00:00
4,2024,26729,0,19,0.0711,13784,51.5695,8369,31.3106,5782,21.6319,3619,13.5396,2024-01-01 12:03:00+00:00,2024-12-31 23:04:00+00:00
5,2025,26186,0,0,0.0000,12786,48.8276,7211,27.5376,4896,18.6970,2972,11.3496,2025-01-01 01:43:00+00:00,2025-12-31 22:47:00+00:00
6,2026,27,0,0,0.0000,11,40.7407,5,18.5185,4,14.8148,3,11.1111,2026-01-01 02:06:00+00:00,2026-01-01 17:17:00+00:00


,year,source_report_count,candidate_reported_incident_count,collapsed_duplicate_report_count,exception_count,pair_decision_incident_count,pair_decision_count,uncertain_pair_decision_count
0,2020,10402,9840,562,0,3453,5531,3388
1,2021,21663,19653,2010,0,5528,8874,7635
2,2022,30799,28270,2529,0,7900,15221,13492
3,2023,19329,18171,1157,1,3930,6805,6280
4,2024,26729,25140,1586,3,7246,11727,10064
5,2025,26186,24573,1613,0,6413,9121,8099
6,2026,27,26,1,0,5,3,1


{'status': 'complete',
 'claim_boundary': ['Date/Time is a user-entered reported incident date and time supplied as UTC for this pipeline.',
  'Crossing-local time is derived from current Form 71 coordinates and historical IANA offsets.',
  'Available evidence does not prove submission time, exact physical start time, or second-accurate observation time.',
  'Duration is a documented proxy interval, not a measured or verified incident endpoint.',
  'Pair decisions are deterministic ruleset outputs, not manual labels or ground truth about physical events.',
  'An interval without a report can be labeled no_report_observed, not unblocked.'],
 'summary': {'ruleset_version': 'phase1-v3',
  'source_reports': 135135,
  'candidate_reported_incidents': 125673,
  'exceptions': 4,
  'auto_merged_reports': 9458,
  'consolidation_tiers': {'distinct_candidate': 120051,
   'overlap_proxy_non_temporal_match': 4723,
   'exact': 888,
   'normalized_exact': 11},
  'pair_decisions': 57282,
  'pair_decisi